In [ ]:
# This notebook will cover part 1.13 in the llm-zoomcamp course, I need to create this
# separate because in the main llm-zoomcamp files,
# I have turned search into database rather than minsearch, and it is much
# simpler to illustrate turning RAG into agentic RAG using minsearch. 

# The first half of this notebook will be just the older minsearch stuff, I will create a divider 
# below from when the agentic code starts.

In [123]:
from openai import OpenAI
openai_client = OpenAI()

In [124]:
from dotenv import load_dotenv
load_dotenv()

True

In [125]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [126]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Usually, yes — if the course is still open and hasn’t passed the enrollment deadline, you can join after it has started.

A few things to check:
- **Enrollment status**: Is late registration allowed?
- **Course progress**: Have you missed key intro lessons, assignments, or deadlines?
- **Access**: Can you still get the materials and class recordings?
- **Instructor approval**: Some courses require permission to join late.

If you want, I can help you draft a short message to the instructor asking if you can still enroll.


In [127]:
context = """I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active."""

In [128]:
prompt = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"

Question: 
{question}

Context:
{context}'''

In [129]:
print(prompt)

Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"

Question: 
I just discovered the course. Can I join now?

Context:
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors

In [130]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [131]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [132]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [133]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [134]:
"""
We start with R in RAG, which is Retrieval, but he calls it SEARCH.
"""

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [135]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
index.search(question)  # returns questions from all of the courses, llm_zoomcamp, AI Dev Zoomcamp, Data Engineering Zoomcamp, etc.

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
 

In [136]:
index.search(question, filter_dict={'course':'llm-zoomcamp'})  # returns all the questions only from the llm-zoomcamp course

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [137]:
search_results =index.search(question, filter_dict={'course':'llm-zoomcamp'}, num_results=5) # if we just want 5 results

In [138]:
# this was our original RAG function
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)

# We have now covered the search function, so let's add it
def search(query, course='llm-zoomcamp'):
    boost_dict={'question':2.0, 'section':0.5}
    filter_dict={'course':course}
    
    return index.search(query,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [139]:
search_results = search(question)  # note that this search() function is the minsearch's Index() function and not
                                   # the search() function we defined just above.
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [140]:
"""
We move on to the A in RAG, which is Augmented, but he calls it PROMPT.


He said that Prompt is split into two parts: 

1) Instructions 
2) User prompt

So let's split up what we had towards the beginnig of all of this.
"""

"\nWe move on to the A in RAG, which is Augmented, but he calls it PROMPT.\n\n\nHe said that Prompt is split into two parts: \n\n1) Instructions \n2) User prompt\n\nSo let's split up what we had towards the beginnig of all of this.\n"

In [141]:
INSTRUCTIONS = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant informaitona nd provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"
'''

In [142]:
USER_PROMPT_TEMPLATE = '''
Question: 
{question}

Context:
{context}
'''

In [143]:
# Next up, we turn the python dictionaries that we got from our search into one long string, this makes it
# easier for the LLM to read and understand.
# I'll copy over a function that I wrote before, it takes the search_results which is the two json dicitonaries 
# and it turns them into one long string. 

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["course"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip() 

build_context(search_results)



'llm-zoomcamp\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nllm-zoomcamp\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nllm-zoomcamp\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nllm-zoomcamp\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chain

In [144]:
# Then we feed this long string into our prompt template that we saw before, that we will send to the llm. 
# Say search_result we got from part one, build_context that turns it into a long string 
# and the prompt, which simply formats the long string using the USER_PROMPT_TEMPLATE. 
# And we remove any trailing or leading whitespace with strip. 
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


prompt = build_prompt(question, search_results)

print(prompt)

Question: 
I just discovered the course. Can I join now?

Context:
llm-zoomcamp
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

llm-zoomcamp
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

llm-zoomcamp
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

llm-zoomcamp
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, 

In [145]:
"""
Then the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, 
hence generation.
"""

'\nThen the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, \nhence generation.\n'

In [146]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [147]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [148]:
# We can now put this together into an updated llm function.
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


In [149]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, you need to submit your project while submissions are still being accepted.


In [150]:
# ---------------------------------------------
# The above all minsearch RAG stuff we have seen before, below is agentic code

In [151]:
# When you have a typo in Ollama and you send it into our lexiographical text minsearch function
rag('how do I run Olama?')

'I do not know'

In [152]:
# When you send it directly to ChatGPT, i.e. call the llm directly with this 
search('how do I run Olama?')   

[{'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the AI techniques the course is teaching: AI Copilot to generate flows from natural language, RAG to ground responses in real data, and AI agents that decide which tools to call. The goal is to see how AI fits into production-style workflows, not just notebook cells.\n\nKestra's AI plugins also work with any major provider (OpenAI, Gemini, Anthropic, and more), so you can swap providers in a flow without changing anything else. See the [module intro](https://github.com/DataTalksClub/llm-zoom

In [153]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [154]:
messages = [
        {"role": "user", "content": "I just discovered the course. Can I join it?"}
    ]

In [155]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

In [156]:
# Notice how the llm has slightly changed the wording of the question? - this is what we want 
# because if we have a typo very likely it will correct it for us!
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after start registration deadline FAQ"}', call_id='call_k3ePIioWMmCKzuKnh6RQoSD2', name='search', type='function_call', id='fc_04be324f59860a6b006aba328e192c87d2b1555bce7a1df7cc', async_=None, caller=None, namespace=None, status='completed')]

In [ ]:
# So we assign the output of the llm to 'call'.
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after start registration deadline FAQ"}', call_id='call_k3ePIioWMmCKzuKnh6RQoSD2', name='search', type='function_call', id='fc_04be324f59860a6b006aba328e192c87d2b1555bce7a1df7cc', async_=None, caller=None, namespace=None, status='completed')

In [158]:
# so call.arguments points at the query that the llm has 'changed' our initial prompt to!
call.arguments

'{"query":"join course discovered late enrollment can I join after start registration deadline FAQ"}'

In [159]:
import json 

args = json.loads(call.arguments)

In [160]:
results = search(**args)

In [161]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'cdc3b285e5',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I submit homework after the deadline, or get a deadline extension?',
  'answer': "No. We don't give individual deadline extensions, and once the homework submission form is closed you can no longer submit it — there are no late submissions. While the form is still open you can submit, even if the listed deadline has already passed.\n\nMissing a homework won't affect your certificate: homework isn't mandatory, only passing the Capstone project is. Homework points only count toward your leaderboard rank, so you'll still appear on the leaderboard with your other submissions."},


In [162]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "cdc3b285e5",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Can I submit homework after the deadline, or get a deadline extension?",
    "answer": "No. We don't give individual deadline extensions, and once the homework submission form is closed you can no longer submit it \u2014 there are no late submissions. While the form is still open you can submit, even if the listed deadline has already passed.\n\nMissing a homework won't affect your certificate: homework isn't mandatory, only passing the Capstone project is. Homework points only count toward your leaderboard rank, so you'll still appear on the l

In [ ]:
# We format the above output in a way such that we can plug it into the API call to OpenAI further down.

# IMPORTANTLY, function_call_output is my program's REPLY to the request the model made. 
# The name is a bit confusing, because "output" here refers to the output of the function, 
# i.e. what search() returned. It is NOT the llm's output.

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [170]:
# IMPORTANTLY, if we try it. See how it throws an error?

response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=function_call_output,
        tools=[search_tool]
    )

BadRequestError: Error code: 400 - {'error': {'message': "Invalid type for 'input': expected one of a string or array of input items, but got an object instead.", 'type': 'invalid_request_error', 'param': 'input', 'code': 'invalid_type'}}

In [ ]:
# It throws an error and this is important. 
# LLM's are STATELESS, so they need to know the history. 
# In our case, this is what we have: 

"""
1. make a call to the LLM <--
2. LLM decide to invoke search ('params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (another call to LLM) <-- 
5. LLM processes the results 
6. LLM gives the answer

In out case, when we hit the error above, we are calling the LLM the second time at 4! 

So we need to feed it the history (in this case when we called it at step 1.); when the llm 
rephrased our question.
"""

In [165]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [ ]:
# Note call is the response from the first call we 
# made to the llm
messages.append(call)

In [167]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after start registration deadline FAQ"}', call_id='call_k3ePIioWMmCKzuKnh6RQoSD2', name='search', type='function_call', id='fc_04be324f59860a6b006aba328e192c87d2b1555bce7a1df7cc', async_=None, caller=None, namespace=None, status='completed')]

In [168]:
messages.append(function_call_output)

In [171]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after start registration deadline FAQ"}', call_id='call_k3ePIioWMmCKzuKnh6RQoSD2', name='search', type='function_call', id='fc_04be324f59860a6b006aba328e192c87d2b1555bce7a1df7cc', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_k3ePIioWMmCKzuKnh6RQoSD2',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "cdc3b285e5",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Can I submit homework afte

In [169]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool]
    )

print(response.output_text)

Yes, you can still join the course.

If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.


In [ ]:
# Pretty important - I didn't get why we appended both call and function_call output to messages: 

#CLAUDE's explanation: 
"""
The list is a transcript, not a log of your API calls

Every item in messages is one 'turn' by one participant (participants being myself and the LLM):

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"},   # 1. the human asks
    call,                                                                          # 2. the model: "run search(...)"
    function_call_output,                                                          # 3. the tool: "here's what I found"
]
# → you now ask the model for turn 4: the actual answer


Turn 2 is a question the model asked you. Turn 3 is your answer to it. 

You have to append the output for the same reason you had to append the user message: without it, the transcript contains a request that was never answered, and the model has nothing to base an answer on. 
"""